# 📱 Actividad — Regresión logística (Phone Classification)

## 🛠️ Funciones auxiliares

Ejecuta la siguiente celda **una sola vez**. Después sólo llama a las funciones. Son las de los laboratorios de regresión logística y árboles; `evaluar_clasificador` y `plot_matriz_confusion` funcionan con más de dos clases.

| Función | ¿Para qué sirve? |
|---|---|
| `plot_frequencies(df, columns, top_n=None)` | Frecuencia de cada categoría |
| `plot_correlation_matrix(df, columns)` | Correlación de Pearson entre variables numéricas |
| `escalar(X_train, X_test)` | Estandariza (media 0, desviación 1) ajustando sólo con train |
| `plot_rfecv_clasificacion(rfecv, metrica='exactitud')` | Desempeño de validación cruzada según las variables que conserva RFECV |
| `evaluar_clasificador(nombre, y_true, y_pred)` | Exactitud, precisión, recall y F1 (macro si hay más de dos clases) |
| `plot_matriz_confusion(y_true, y_pred, clases=None, etiquetas=None)` | Matriz de confusión; la diagonal son los aciertos |


In [ ]:
# Funciones auxiliares — Caso 2: Regresión logística (Phone Classification)
# Son las mismas que usamos en los laboratorios de regresión logística y árboles.
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   evaluar_clasificador y plot_matriz_confusion se ajustaron para funcionar
#      con MÁS DE DOS clases (en Titanic sólo había dos: sobrevivió / no sobrevivió).
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.preprocessing import StandardScaler


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def escalar(X_train, X_test):
    """Estandariza (media 0, desviación 1) ajustando sólo con train. Regresa ambos DataFrames."""
    escalador = StandardScaler().fit(X_train)
    X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_s, X_test_s


def plot_rfecv_clasificacion(rfecv, metrica='exactitud'):
    """Desempeño promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name=f'{metrica} promedio (CV)'
    ))
    fig.add_vline(x=rfecv.n_features_, line_dash='dash', line_color='red',
                  annotation_text=f'elegido: {rfecv.n_features_}')
    fig.update_layout(
        title='RFECV — desempeño según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title=f'{metrica.capitalize()} (validación cruzada)',
        template='plotly_white', width=900, height=450
    )
    fig.show()


def evaluar_clasificador(nombre, y_true, y_pred):
    """Imprime y regresa exactitud, precisión, recall y F1.

    Con más de dos clases, precisión, recall y F1 son el promedio simple
    de las clases (average='macro').
    """
    binario = len(np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)]))) <= 2
    promedio = 'binary' if binario else 'macro'
    metricas = {
        'modelo': nombre,
        'exactitud': round(accuracy_score(y_true, y_pred), 4),
        'precisión': round(precision_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'recall': round(recall_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'F1': round(f1_score(y_true, y_pred, average=promedio, zero_division=0), 4),
    }
    print(f"{nombre}: exactitud {metricas['exactitud']} | precisión {metricas['precisión']} "
          f"| recall {metricas['recall']} | F1 {metricas['F1']}")
    return metricas


def plot_matriz_confusion(y_true, y_pred, clases=None, etiquetas=None, title='Matriz de confusión'):
    """Aciertos y errores del clasificador: la diagonal son los aciertos.

    clases: los valores de la variable objetivo en el orden en que quieres verlos,
            p. ej. ['Low', 'Mid_Low', 'Mid', 'High']. Si no las pasas, se ordenan alfabéticamente.
    etiquetas: nombres a mostrar para esas clases (opcional; por defecto, las mismas clases).
    """
    if clases is None:
        clases = sorted(set(np.asarray(y_true)) | set(np.asarray(y_pred)))
    clases = list(clases)
    etiquetas = list(etiquetas) if etiquetas is not None else clases

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=clases),
        index=[f'Real: {e}' for e in etiquetas],
        columns=[f'Predicho: {e}' for e in etiquetas]
    )

    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title=title)
    fig.update_layout(width=650, height=450)
    fig.show()
    return cm


---
# 📥 Datos

Bob quiere predecir el **rango de precio** de un celular (`Low`, `Mid_Low`, `Mid`, `High`) a partir de sus características (RAM, memoria interna, cámara, wifi, etc.). La variable dependiente es `price_range`.


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/phone_classification.csv"
df = pd.read_csv(url)
df.head()


---
# 1️⃣ Exploración de datos

## Cantidad de observaciones por `price_range`

Gráfico de barras con Plotly: cuántos celulares hay en cada rango de precio.


In [ ]:
# Cantidad de observaciones por price_range (Plotly)
orden = ['Low', 'Mid_Low', 'Mid', 'High']
conteo = (
    df['price_range']
    .value_counts()
    .reindex(orden)
    .rename_axis('price_range')
    .reset_index(name='cantidad')
)

fig = px.bar(
    conteo,
    x='price_range',
    y='cantidad',
    text='cantidad',
    title='Cantidad de observaciones por price_range',
    labels={'cantidad': 'Cantidad de observaciones', 'price_range': 'price_range'},
)
fig.update_layout(
    xaxis={'categoryorder': 'array', 'categoryarray': orden},
    template='plotly_white',
)
fig.show()
